# Clase 7 — Storytelling: los datos no convencen solos

**Duración estimada: 90 minutos**

## Objetivos de aprendizaje

- Explicar por qué contar una historia con los datos es tan importante como analizarlos.
- Convertir una tabla de resultados en una visualización que comunique una conclusión clara.
- Aplicar principios básicos de storytelling de datos a una presentación propia.

Este notebook es autocontenido: vuelve a cargar y limpiar el dataset de recetas (ya lo hiciste en clases anteriores) y luego construye, con código nuevo, el material de este capítulo. Consulta el [capítulo 7 del manual](../manual/08_storytelling.md) para la explicación completa en prosa.

## 0. ¿Por qué un capítulo entero sobre esto?

Este tema no es una etapa de la metodología de Rollins ni de CRISP-DM. No aparece en ningún diagrama de las 10 etapas. Y aun así, tiene su propio capítulo porque es la habilidad que la metodología da por sentada sin nombrarla: comunicar lo que encontraste. Se apoya en todo lo anterior — en particular, en la visualización, que ya usaste en la Clase 4 para *explorar* datos y que hoy vas a usar para *comunicarlos*.

## 1. La frase que resume el problema

> "Todo el mundo puede mostrar números, pero si no tienes una historia detrás, si no tienes una razón convincente para actuar, al final lo que presentas no va a conectar con tu audiencia."

Los humanos entendemos el mundo a través de **historias**, no a través de tablas de números. Una tabla de 15 filas y 8 columnas puede ser perfectamente correcta y, para la mayoría de la gente que la recibe, completamente invisible.

## 2. El estudio de Stanford

En Stanford se hizo un estudio con dos formas de presentar el mismo pitch (presentación breve para convencer de algo):

- **Grupo A**: presentó solo KPIs (indicadores clave de rendimiento), cifras y estadísticas.
- **Grupo B**: presentó esos mismos datos, pero envueltos en una historia.

Después se preguntó a la audiencia qué recordaba de cada presentación. **Lo que se quedó en la memoria fueron las historias.** Las cifras seguían dentro de esas historias — no es que los números sobraran, es que solos no se recordaban.

**Implicación práctica:** si presentas una tabla y esperas que la conclusión se recuerde, apuestas contra cómo funciona la memoria humana. Si la conviertes en una historia clara, tienes muchas más probabilidades de que se traduzca en una decisión real.

## 3. El último kilómetro de la entrega

*"The last mile in delivery"*: la parte técnica (cargar, limpiar, modelar, evaluar) se aprende en un periodo de formación relativamente corto. Lo que escasea es la capacidad de **extraer valor de los datos y comunicarlo** de forma que alguien actúe. Para una carrera larga en esta disciplina, esta es la habilidad que marca la diferencia.

Y tiene una segunda cara: el storytelling también sirve **al propio analista**. Contarte a ti mismo una historia sobre un dataset mientras trabajas con él te ayuda a entenderlo mejor — es una herramienta de comprensión, no solo de presentación final.

## 4. Simplicidad contra complejidad: una tensión real

Contar una historia simple y clara, sin perder los matices importantes de los datos, **es una tensión difícil, no un problema con truco**. Cuanto más simplificas, más fácil de recordar — pero más fácil que escondas algo importante. Cuanto más fiel eres a la complejidad, más honesta la historia — pero menos memorable.

No hay una fórmula que la resuelva de una vez. Es un equilibrio que negocias cada vez que presentas algo, según quién te escucha y qué decisión está en juego.

## 5. De exploración a comunicación

En la Clase 4 usaste tablas y estadística descriptiva para **explorar** el dataset de recetas — para que tú, como analista, entendieras qué había ahí antes de limpiarlo y modelarlo. Eso es EDA (análisis exploratorio de datos): mirar los datos sin audiencia, solo para ti.

Hoy vas a usar la visualización con un propósito distinto: no para que tú entiendas, sino para que **otra persona, sin tocar el dataset, se lleve una conclusión clara en segundos**. Misma herramienta (un gráfico), otro objetivo.

## 6. Recordatorio: limpieza mínima del dataset de recetas

Esto ya lo hiciste en clases anteriores. Lo repetimos aquí para que el notebook sea autocontenido.

In [ ]:
import pandas as pd
import numpy as np

recipes = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv")

column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names
recipes["cuisine"] = recipes["cuisine"].str.lower()

recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

recipes.shape

In [ ]:
# El mismo cálculo del capítulo 4: cuántas recetas contiene cada ingrediente
ing = recipes.iloc[:, 1:].sum(axis=0)
ingredient = pd.Series(ing.index.values, index=np.arange(len(ing)))
count = pd.Series(list(ing), index=np.arange(len(ing)))
ing_df = pd.DataFrame(dict(ingredient=ingredient, count=count))
ing_df = ing_df[["ingredient", "count"]]
ing_df.sort_values(["count"], ascending=False, inplace=True)
ing_df.reset_index(inplace=True, drop=True)

ing_df.head(10)

## 7. La tabla como está: ¿te acuerdas de algo?

Mira la tabla anterior 10 segundos y luego cierra los ojos. ¿Qué recuerdas? Probablemente "egg" y poco más, y solo porque está primero. Eso es exactamente el problema que describe el estudio de Stanford: la tabla es correcta, completa, y no se queda en la memoria. Vamos a convertir la misma información — sin cambiar ni un número — en una historia visual.

## 8. De tabla a historia: el gráfico de barras horizontal

Un gráfico de barras horizontal es más legible que una tabla de texto larga por varias razones:

- Los nombres de ingrediente son texto, y en horizontal se leen de corrido sin girar etiquetas.
- La longitud de la barra comunica la magnitud de un vistazo, sin que tengas que comparar cifras mentalmente.
- Ordenado de mayor a menor con el más frecuente arriba, el ojo capta la conclusión en el primer segundo.

Nos quedamos con los 10 ingredientes más frecuentes: es la parte del ranking que de verdad le importa a una audiencia, el resto es ruido para este propósito.

In [ ]:
import matplotlib.pyplot as plt

top10 = ing_df.head(10)

plt.figure(figsize=(8, 5))

# barh() = barras horizontales. Mejor que un gráfico vertical cuando
# las etiquetas de categoría son texto largo (nombres de ingredientes).
plt.barh(top10["ingredient"], top10["count"], color="steelblue")

# invert_yaxis() deja el ingrediente más frecuente arriba del todo:
# es donde el ojo humano mira primero al leer un gráfico.
plt.gca().invert_yaxis()

# El título no describe el gráfico ("Top 10 ingredientes"), dice la
# conclusión directamente. Esa es la diferencia entre tabla e historia.
plt.title("Egg, wheat y butter dominan el dataset (sesgado por ~40.000 recetas americanas)")
plt.xlabel("Número de recetas que lo contienen")
plt.ylabel("Ingrediente")

plt.tight_layout()
plt.show()

Fíjate: los datos son idénticos a los del capítulo 4 (egg 21.025, wheat 20.781, butter 20.719...). Lo único que cambió es la forma en la que llegan a quien los mira. Y, siguiendo el principio de "simplifica sin mentir", el título también recuerda el sesgo (las ~40.000 recetas americanas) en vez de esconderlo — no hace falta ponerlo todo en el gráfico, pero tampoco ocultarlo.

## 9. Tabla vs. gráfico, lado a lado

Para que veas el contraste de impacto de un vistazo, aquí tienes la misma información dos veces: primero como texto plano (lo que verías en una tabla dentro de un informe), y justo debajo, como gráfico.

In [ ]:
# Versión "tabla de texto plano", como aparecería en un informe sin gráficos
print("Ranking de ingredientes más frecuentes (tabla de texto):\n")
for _, row in ing_df.head(10).iterrows():
    print(f"{row['ingredient']:<15} {row['count']:>8.0f}")

In [ ]:
# Misma información, como historia visual
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10["ingredient"], top10["count"], color="steelblue")
ax.invert_yaxis()
ax.set_title("Los 10 ingredientes más comunes del dataset de recetas")
ax.set_xlabel("Número de recetas que lo contienen")
plt.tight_layout()
plt.show()

Ambas celdas muestran exactamente los mismos 10 ingredientes y los mismos conteos. Pero solo una de las dos es la que recordarías si las vieras 5 segundos en una presentación y pasaras a la siguiente diapositiva. Esa es la idea central de este capítulo: el storytelling no cambia el hallazgo, cambia si ese hallazgo sobrevive fuera de la pantalla.

## 10. Principios prácticos, derivados de todo lo anterior

- **Empieza por la conclusión, no por el método.** El título de un gráfico debería decir algo por sí solo, no describir los ejes.
- **Elige un gráfico en vez de una tabla cuando quieras que algo se recuerde.** La tabla es para que tú explores; el gráfico es para que otra persona recuerde.
- **Simplifica sin mentir.** No escondas la complejidad importante (como el sesgo americano), pero tampoco la pongas toda encima del gráfico principal.
- **Dale a tu audiencia una razón para actuar, no solo información.** La diferencia entre "esto es lo que hay" y "esto es lo que deberíamos hacer" es, muchas veces, la diferencia entre una presentación que se olvida y una que cambia una decisión.

## Resumen

- Los humanos recordamos historias, no tablas de números; el estudio de Stanford lo confirmó comparando presentaciones con solo KPIs frente a presentaciones con historia.
- El storytelling es "el último kilómetro en la entrega": lo técnico se aprende rápido, comunicar valor es escaso y decisivo para una carrera larga.
- Existe una tensión real, sin solución mágica, entre simplicidad y complejidad al contar una historia con datos.
- El storytelling también sirve al propio analista para entender mejor el dataset mientras trabaja con él.
- La visualización no cambia entre exploración (Clase 4) y comunicación (esta clase): cambia el propósito con el que la usas.
- Convertir `ing_df` en un `plt.barh()` de los 10 primeros, con el más frecuente arriba y un título que dice la conclusión, es el mismo hallazgo del capítulo 4 contado como historia en vez de como tabla.

En el notebook de práctica vas a aplicar esto mismo a una cocina concreta: elegir sus ingredientes característicos y convertirlos en un gráfico con un título que cuente una conclusión, no solo un ranking.